# 2 · Price elasticity
**Model:** `log(units) = e·log(price) + b1·feature + b2·display + (store x product) + month + trend`. Each product in each store has its own normal level, so `e` comes only from price moves over time; the assortment differs by store, which is why store and product are not separate effects.

`e` is the elasticity: +1% price changes units by about `e`%. Example: e = -1.8 means +1% price, about -1.8% units, so revenue falls.

It is a **correlation on historical data**, not an experiment.

In [1]:
import os, sys
from pathlib import Path
# run from the repository root so relative paths work
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import pandas as pd, numpy as np
pd.set_option("display.width", 160, "display.max_columns", 20)

In [2]:
import elasticity as el_mod
df = el_mod.load()
print(len(df), 'clean rows')

522624 clean rows


## One regression in full: the most-sold category
The price coefficient is the elasticity; feature and display are controls so their volume is not credited to the price cut.

In [3]:
top = df.groupby('category')['units'].sum().idxmax()
res = el_mod.fit(df[df['category'] == top])
print(top, '- R2', round(res.rsquared, 3), '- rows', int(res.nobs))
pd.DataFrame({'coef': res.params, 'ci_low': res.conf_int()[0], 'ci_high': res.conf_int()[1], 'p': res.pvalues}).loc[['log_price', 'feature', 'display', 'trend']].round(3)

Cold Cereal - R2 0.442 - rows 169498


,coef,ci_low,ci_high,p
log_price,-1.608,-1.662,-1.554,0.000
feature,0.483,0.450,0.516,0.000
display,0.549,0.521,0.576,0.000
trend,-0.015,-0.027,-0.003,0.014


## All segments (business rule BR-06 decides what is published)

In [4]:
e = pd.read_csv('data/output/elasticity.csv')
e[e['level'].isin(['All', 'Category', 'Brand'])][['level', 'segment', 'n_obs', 'elasticity', 'ci_low', 'ci_high', 'status', 'status_reason']]

,level,segment,n_obs,elasticity,ci_low,ci_high,status,status_reason
0,All,All categories (pooled),522624,-1.613,-1.668,-1.558,Published,NaN
1,Category,Bag Snacks,126187,-1.228,-1.308,-1.148,Published,NaN
2,Brand,Bag Snacks | Frito Lay,32067,-1.516,-1.707,-1.325,Published,NaN
3,Brand,Bag Snacks | Mksl,13699,-1.488,-1.734,-1.242,Published,NaN
4,Brand,Bag Snacks | Private Label,34347,-0.608,-0.737,-0.478,Published,NaN
5,Brand,Bag Snacks | Shultz,14618,-0.310,-0.411,-0.209,Published,NaN
6,Brand,Bag Snacks | Snyder's,31456,-1.835,-1.969,-1.701,Published,NaN
11,Category,Cold Cereal,169498,-1.608,-1.662,-1.554,Published,NaN
12,Brand,Cold Cereal | General Mills,35898,-0.780,-0.820,-0.741,Published,NaN
13,Brand,Cold Cereal | Kellogg,35790,-2.020,-2.122,-1.917,Published,NaN


## Sanity checks
A positive and significant elasticity is suspicious: it goes to review, never to the simulator.

In [5]:
e.groupby(['level', 'status']).size().unstack(fill_value=0)

status,Insufficient data,Published,Review
level,,,
All,0,1,0
Brand,0,17,2
Category,0,4,0
Market,8,8,0
Product,0,51,4


## Regular price vs promotion price cut, and cross-elasticities

In [6]:
pd.read_csv('data/output/elasticity_split.csv')

,category,component,elasticity,ci_low,ci_high,p_value,n_obs,base_price_sd,status,note
0,Bag Snacks,Regular price,-1.606,-1.727,-1.486,0.0,126187,0.111,Estimated,NaN
1,Bag Snacks,Promo price cut,-1.002,-1.078,-0.926,0.0,126187,0.111,Estimated,NaN
2,Cold Cereal,Regular price,-1.698,-1.800,-1.595,0.0,169498,0.071,Estimated,NaN
3,Cold Cereal,Promo price cut,-1.581,-1.631,-1.532,0.0,169498,0.071,Estimated,NaN
4,Frozen Pizza,Regular price,-2.794,-2.927,-2.661,0.0,110766,0.087,Estimated,NaN
5,Frozen Pizza,Promo price cut,-2.899,-3.061,-2.738,0.0,110766,0.087,Estimated,NaN
6,Oral Hygiene Products,Regular price,-1.180,-1.250,-1.110,0.0,116173,0.088,Estimated,NaN
7,Oral Hygiene Products,Promo price cut,-1.285,-1.343,-1.227,0.0,116173,0.088,Estimated,NaN


In [7]:
pd.read_csv('data/output/cross_elasticity.csv')

,category,manufacturer,n_obs,own_elasticity,cross_elasticity,ci_low,ci_high,p_value,reading
0,Bag Snacks,Frito Lay,11807,-2.289,-0.059,-0.199,0.081,0.4098,no clear link
1,Bag Snacks,Mksl,5000,-2.648,0.582,0.269,0.895,0.0003,substitute
2,Bag Snacks,Private Label,11908,-0.961,0.261,0.061,0.460,0.0104,substitute
3,Bag Snacks,Shultz,5151,-0.963,0.159,-0.157,0.475,0.3231,no clear link
4,Bag Snacks,Snyder's,11670,-0.448,0.255,0.110,0.400,0.0006,substitute
5,Cold Cereal,General Mills,11987,-1.318,-0.050,-0.108,0.007,0.0879,no clear link
6,Cold Cereal,Kellogg,11987,-2.251,0.975,0.882,1.067,0.0000,substitute
7,Cold Cereal,Post Foods,10788,-1.690,0.155,0.057,0.254,0.0020,substitute
8,Cold Cereal,Private Label,11987,-1.207,0.302,0.240,0.364,0.0000,substitute
9,Cold Cereal,Quaker,10778,-2.509,-1.194,-1.387,-1.000,0.0000,complement?
